# Scraping — Bonheur & Éducation par pays

**Objectif :** Constituer un dataset CSV labelisé, pays par pays, combinant :
- **Scores de bonheur** (World Happiness Report)
- **Indices d'éducation** (Education Index, taux d'alphabétisation, scolarisation)

**Pipeline :** Fetching → Parsing → Extracting → Transforming

**Sources :**
| Source | Données |
|---|---|
| Wikipedia — Education Index | Indice d'éducation HDI par pays |
| Wikipedia — World Happiness Report | Score & rang de bonheur par pays |
| NationMaster | Taux d'alphabétisation, scolarisation |
| Wikipedia — Literacy by country | Taux d'alphabétisation de secours |

## 0. Setup

In [ ]:
# !pip install requests beautifulsoup4 pandas numpy lxml html5lib

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time
from io import StringIO

# Headers qui imitent un vrai navigateur (évite les blocages 403)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def fetch(url: str, delay: float = 1.0) -> BeautifulSoup:
    """Fetche une URL et retourne un objet BeautifulSoup.
    
    Respecte un délai entre les requêtes pour éviter de surcharger les serveurs.
    Lève une exception si la requête échoue.
    """
    time.sleep(delay)
    response = SESSION.get(url, timeout=15)
    response.raise_for_status()  # Lève HTTPError si statut >= 400
    return BeautifulSoup(response.text, "lxml")

print("Setup OK")

---
## 1. Education Index — Wikipedia

**URL :** https://en.wikipedia.org/wiki/Education_Index

La page contient un tableau HDI Education Index avec une valeur entre 0 et 1 par pays.

### 1.1 Fetching

In [ ]:
URL_EDU_INDEX = "https://en.wikipedia.org/wiki/Education_Index"

soup_edu = fetch(URL_EDU_INDEX)
print(f"Page fetchée : {soup_edu.title.text}")
print(f"Taille HTML  : {len(soup_edu.prettify()):,} caractères")

### 1.2 Parsing — Localisation du tableau

In [ ]:
# Wikipedia utilise la classe "wikitable" pour ses tableaux de données
tables_edu = soup_edu.find_all("table", class_="wikitable")
print(f"Tableaux wikitable trouvés : {len(tables_edu)}")

# Inspection des en-têtes pour identifier le bon tableau
for i, tbl in enumerate(tables_edu):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")[:6]]
    print(f"  Tableau {i} : {headers}")

### 1.3 Extracting

In [ ]:
def extract_wikitable(table_tag) -> pd.DataFrame:
    """Extrait un <table> Wikipedia en DataFrame pandas.
    
    Gère les cellules fusionnées (rowspan/colspan) fréquentes sur Wikipedia.
    """
    html_str = str(table_tag)
    # pd.read_html parse les tableaux HTML directement
    dfs = pd.read_html(StringIO(html_str), flavor="lxml")
    return dfs[0]


def clean_text(val) -> str:
    """Supprime les annotations Wikipedia entre crochets et les espaces superflus."""
    return re.sub(r"\[.*?\]", "", str(val)).strip()


def to_float(val) -> float:
    """Convertit une valeur textuelle en float, NaN si impossible."""
    try:
        cleaned = re.sub(r"[^\d.]", "", clean_text(val))
        return float(cleaned)
    except (ValueError, TypeError):
        return np.nan


# Extraction du tableau principal (le premier wikitable contient l'index)
df_edu_raw = extract_wikitable(tables_edu[0])
print("Colonnes brutes :", df_edu_raw.columns.tolist())
df_edu_raw.head()

### 1.4 Transforming — Education Index

In [ ]:
def normalize_country(name: str) -> str:
    """Normalise un nom de pays pour faciliter les jointures inter-sources.
    
    - Supprime les annotations Wikipedia
    - Strip des espaces
    - Harmonise les variantes d'orthographe courantes
    """
    name = clean_text(name)
    aliases = {
        "United States": "United States of America",
        "United States of America": "United States of America",
        "USA": "United States of America",
        "UK": "United Kingdom",
        "Russia": "Russia",
        "Russian Federation": "Russia",
        "Korea, South": "South Korea",
        "Korea, North": "North Korea",
        "Republic of Korea": "South Korea",
        "Democratic People's Republic of Korea": "North Korea",
        "Congo, Dem. Rep.": "Democratic Republic of the Congo",
        "Congo, Rep.": "Republic of the Congo",
        "Iran, Islamic Rep.": "Iran",
        "Lao PDR": "Laos",
        "Kyrgyz Republic": "Kyrgyzstan",
        "Slovak Republic": "Slovakia",
        "Czechia": "Czech Republic",
        "Cabo Verde": "Cape Verde",
        "Eswatini": "Swaziland",
        "North Macedonia": "Macedonia",
        "Viet Nam": "Vietnam",
        "Côte d'Ivoire": "Ivory Coast",
    }
    return aliases.get(name, name)


# Identification des colonnes pertinentes.
# Le tableau Wikipedia Education Index a pour colonnes : Country | 2019 | 2018 | ... | 1990
# Les colonnes numériques sont des années — on prend la plus récente comme valeur de référence.
col_map_edu = {}
year_cols = []

for col in df_edu_raw.columns:
    col_str = str(col).strip()
    col_lower = col_str.lower()
    if "country" in col_lower or "nation" in col_lower:
        col_map_edu["country"] = col
    elif "rank" in col_lower:
        col_map_edu["rank_edu"] = col
    elif re.match(r"^\d{4}$", col_str):          # colonne = année (ex: "2019")
        year_cols.append(int(col_str))
    elif "education" in col_lower or "index" in col_lower or "value" in col_lower:
        col_map_edu["education_index"] = col     # fallback si label explicite

# Si les colonnes sont des années, prendre la plus récente
if year_cols and "education_index" not in col_map_edu:
    most_recent_year = str(max(year_cols))
    col_map_edu["education_index"] = most_recent_year
    print(f"Colonnes-années détectées : {sorted(year_cols, reverse=True)[:5]}...")
    print(f"→ Utilisation de l'année {most_recent_year} comme 'education_index'")

print("Mapping colonnes final :", col_map_edu)

df_edu = pd.DataFrame()
df_edu["country"] = df_edu_raw[col_map_edu["country"]].apply(normalize_country)
df_edu["education_index"] = df_edu_raw[col_map_edu["education_index"]].apply(to_float)

if "rank_edu" in col_map_edu:
    df_edu["rank_education_index"] = df_edu_raw[col_map_edu["rank_edu"]].apply(to_float)

# Suppression des lignes sans pays valide
df_edu = df_edu[df_edu["country"].str.strip() != ""].dropna(subset=["education_index"])
df_edu = df_edu.reset_index(drop=True)

print(f"\nEducation Index : {len(df_edu)} pays")
print(f"Valeurs manquantes : {df_edu.isnull().sum().to_dict()}")
df_edu.head(10)

---
## 2. World Happiness Report — Wikipedia

**URL :** https://en.wikipedia.org/wiki/World_Happiness_Report

La page recense les scores du rapport annuel pour chaque pays (Life Ladder score).

### 2.1 Fetching

In [ ]:
URL_HAPPINESS = "https://en.wikipedia.org/wiki/World_Happiness_Report"

soup_happy = fetch(URL_HAPPINESS)
print(f"Page fetchée : {soup_happy.title.text}")
print(f"Taille HTML  : {len(soup_happy.prettify()):,} caractères")

### 2.2 Parsing — Localisation du tableau

In [ ]:
tables_happy = soup_happy.find_all("table", class_="wikitable")
print(f"Tableaux wikitable trouvés : {len(tables_happy)}")

for i, tbl in enumerate(tables_happy):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")[:8]]
    print(f"  Tableau {i} : {headers}")

### 2.3 Extracting + Transforming — Happiness Score

In [ ]:
def extract_happiness_table(tables) -> pd.DataFrame:
    """Sélectionne et extrait le tableau le plus récent du World Happiness Report.
    
    Stratégie : privilégie le tableau contenant 'score' ou 'happiness' dans ses en-têtes.
    """
    best_table = None
    best_score = 0

    for tbl in tables:
        headers_text = " ".join(th.get_text(strip=True).lower() for th in tbl.find_all("th"))
        score = sum([
            ("score" in headers_text) * 3,
            ("happiness" in headers_text) * 2,
            ("country" in headers_text) * 2,
            ("rank" in headers_text),
        ])
        if score > best_score:
            best_score = score
            best_table = tbl

    if best_table is None:
        raise ValueError("Aucun tableau de bonheur trouvé — vérifier la structure de la page.")

    return extract_wikitable(best_table)


df_happy_raw = extract_happiness_table(tables_happy)
print("Colonnes brutes :", df_happy_raw.columns.tolist())
df_happy_raw.head()

In [ ]:
col_map_happy = {}
for col in df_happy_raw.columns:
    col_lower = str(col).lower()
    if "country" in col_lower or "nation" in col_lower:
        col_map_happy["country"] = col
    elif "score" in col_lower or "happiness" in col_lower or "ladder" in col_lower:
        col_map_happy["happiness_score"] = col
    elif "rank" in col_lower:
        col_map_happy["rank_happiness"] = col

print("Mapping colonnes détecté :", col_map_happy)

df_happy = pd.DataFrame()
df_happy["country"] = df_happy_raw[col_map_happy["country"]].apply(normalize_country)
df_happy["happiness_score"] = df_happy_raw[col_map_happy["happiness_score"]].apply(to_float)

if "rank_happiness" in col_map_happy:
    df_happy["rank_happiness"] = df_happy_raw[col_map_happy["rank_happiness"]].apply(to_float)

df_happy = df_happy[df_happy["country"].str.strip() != ""].dropna(subset=["happiness_score"])
df_happy = df_happy.reset_index(drop=True)

print(f"\nHappiness Report : {len(df_happy)} pays")
print(f"Score min : {df_happy['happiness_score'].min():.3f}")
print(f"Score max : {df_happy['happiness_score'].max():.3f}")
df_happy.head(10)

---
## 3. Taux d'alphabétisation — Wikipedia

**URL :** https://en.wikipedia.org/wiki/List_of_countries_by_literacy_rate

Source complémentaire fiable : taux d'alphabétisation total (adultes 15+ ans) par pays.

### 3.1 Fetching + Parsing

In [ ]:
URL_LITERACY = "https://en.wikipedia.org/wiki/List_of_countries_by_literacy_rate"

soup_lit = fetch(URL_LITERACY)
tables_lit = soup_lit.find_all("table", class_="wikitable")
print(f"Tableaux trouvés : {len(tables_lit)}")

for i, tbl in enumerate(tables_lit):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")[:6]]
    print(f"  Tableau {i} : {headers}")

### 3.2 Extracting + Transforming — Literacy Rate

In [ ]:
# Table 1 = répartition par tranche d'âge (Youth/Adult/Elderly) — pas de taux global
# Table 2 = taux total adultes par pays : Country | Total Literacy Rate | Male | Female | ...
# On cherche le tableau qui contient une colonne "total" ou "literacy" dans ses en-têtes.

def find_best_literacy_table(tables) -> pd.DataFrame:
    """Sélectionne le tableau Wikipedia avec le meilleur taux d'alphabétisation global."""
    for tbl in tables:
        try:
            df = extract_wikitable(tbl)
        except Exception:
            continue
        for col in df.columns:
            col_lower = str(col).lower()
            if "total" in col_lower and ("rate" in col_lower or "literacy" in col_lower):
                print(f"→ Tableau retenu, colonne trouvée : '{col}'")
                return df
        # Fallback : au moins "literacy" dans un en-tête
        for col in df.columns:
            if "literacy" in str(col).lower():
                print(f"→ Tableau retenu (fallback), colonne : '{col}'")
                return df
    # Dernier recours : deuxième tableau (index 1)
    print("→ Fallback sur tables_lit[1]")
    return extract_wikitable(tables[1])

df_lit_raw = find_best_literacy_table(tables_lit)
print("Colonnes brutes :", df_lit_raw.columns.tolist())
df_lit_raw.head()

In [ ]:
col_map_lit = {}
for col in df_lit_raw.columns:
    col_lower = str(col).lower()
    if "country" in col_lower or "nation" in col_lower:
        col_map_lit["country"] = col
    elif "total" in col_lower and "literacy_rate" not in col_map_lit:
        col_map_lit["literacy_rate"] = col
    elif ("literacy" in col_lower or "rate" in col_lower) and "literacy_rate" not in col_map_lit:
        col_map_lit["literacy_rate"] = col

print("Mapping colonnes détecté :", col_map_lit)

df_lit = pd.DataFrame()
df_lit["country"] = df_lit_raw[col_map_lit["country"]].apply(normalize_country)
df_lit["literacy_rate_pct"] = df_lit_raw[col_map_lit["literacy_rate"]].apply(to_float)

df_lit = df_lit[df_lit["country"].str.strip() != ""].dropna(subset=["literacy_rate_pct"])
df_lit = df_lit.reset_index(drop=True)

print(f"\nLiteracy Rate : {len(df_lit)} pays")

# --- Diagnostic mismatch ---
# Comparer les noms de pays df_lit vs df_edu (source de référence)
lit_countries = set(df_lit["country"].str.strip().str.lower())
edu_countries = set(df_edu["country"].str.strip().str.lower())
only_in_lit = sorted(
    df_lit[~df_lit["country"].str.strip().str.lower().isin(edu_countries)]["country"].unique()
)
print(f"\nPays dans df_lit absents de df_edu ({len(only_in_lit)}) :")
print(only_in_lit[:30])

df_lit.head(10)

---
## 4. NationMaster — Éducation

**URL :** https://www.nationmaster.com/country-info/stats/Education

NationMaster propose des métriques détaillées (dépenses éducation en % du PIB, ratio enseignants/élèves, etc.).
Le contenu est partiellement rendu côté serveur — on tente d'abord une requête HTTP simple.

### 4.1 Fetching avec tentative de contournement

In [ ]:
URL_NM_EDU = "https://www.nationmaster.com/country-info/stats/Education"

try:
    soup_nm = fetch(URL_NM_EDU, delay=2.0)
    print(f"Statut : OK")
    print(f"Titre  : {soup_nm.title.text if soup_nm.title else 'N/A'}")

    # Détection si la page est rendue (JavaScript) ou statique
    tables_nm = soup_nm.find_all("table")
    stats_links = soup_nm.find_all("a", href=re.compile(r"/country-info/stats/"))
    print(f"Tableaux HTML : {len(tables_nm)}")
    print(f"Liens stats   : {len(stats_links)}")

except requests.HTTPError as e:
    print(f"Erreur HTTP : {e}")
    soup_nm = None
except Exception as e:
    print(f"Erreur : {e}")
    soup_nm = None

### 4.2 Scraping des sous-pages NationMaster (métriques individuelles)

NationMaster organise ses données par métrique, avec une page par indicateur.
On cible les métriques les plus utiles pour l'éducation.

In [ ]:
# Métriques NationMaster ciblées (URLs stables)
NM_METRICS = {
    "gov_expenditure_edu_pct_gdp": "https://www.nationmaster.com/country-info/stats/Education/Government-expenditure-on-education-as-percent-of-GDP",
    "pupil_teacher_ratio_primary": "https://www.nationmaster.com/country-info/stats/Education/Pupil-teacher-ratio%2C-primary",
    "school_enrollment_secondary_pct": "https://www.nationmaster.com/country-info/stats/Education/School-enrollment%2C-secondary-%28percent-gross%29",
}


def scrape_nationmaster_metric(url: str, metric_name: str) -> pd.DataFrame:
    """Scrape une page de métrique NationMaster et retourne un DataFrame (country, metric).
    
    NationMaster affiche les données dans un tableau HTML avec colonnes :
    Rank | Country | Value | Date
    """
    try:
        soup = fetch(url, delay=2.0)
    except requests.HTTPError as e:
        print(f"  [SKIP] {metric_name} — HTTP {e.response.status_code}")
        return pd.DataFrame(columns=["country", metric_name])

    # Chercher le tableau de données principal
    table = soup.find("table")
    if table is None:
        print(f"  [SKIP] {metric_name} — Pas de tableau trouvé (page JS ?)")
        return pd.DataFrame(columns=["country", metric_name])

    rows = []
    for tr in table.find_all("tr")[1:]:  # Skip header
        cells = tr.find_all(["td", "th"])
        if len(cells) < 2:
            continue

        # Structure typique NationMaster : [rank, flag+country, value, ...]
        country_cell = cells[1] if len(cells) > 2 else cells[0]
        value_cell = cells[2] if len(cells) > 2 else cells[1]

        # Extraction du nom de pays (ignorer les images de drapeaux)
        country_text = normalize_country(
            country_cell.get_text(separator=" ", strip=True).split("\n")[0]
        )
        value = to_float(value_cell.get_text(strip=True))

        if country_text:
            rows.append({"country": country_text, metric_name: value})

    df = pd.DataFrame(rows).dropna(subset=[metric_name])
    print(f"  [OK]   {metric_name} — {len(df)} pays")
    return df


# Scraping de toutes les métriques NationMaster
nm_frames = []
for metric_name, url in NM_METRICS.items():
    df_metric = scrape_nationmaster_metric(url, metric_name)
    nm_frames.append(df_metric)

# Fusion des métriques NationMaster en un seul DataFrame
if nm_frames:
    df_nm = nm_frames[0]
    for df_m in nm_frames[1:]:
        if not df_m.empty:
            df_nm = pd.merge(df_nm, df_m, on="country", how="outer")
    print(f"\nNationMaster fusionné : {len(df_nm)} pays, colonnes : {df_nm.columns.tolist()}")
else:
    df_nm = pd.DataFrame(columns=["country"])
    print("NationMaster : aucune donnée collectée")

df_nm.head()

---
## 5. Source complémentaire — Scolarisation UNESCO / Banque Mondiale (Wikipedia)

**URL :** https://en.wikipedia.org/wiki/List_of_countries_by_tertiary_education_attainment

Taux d'accès à l'enseignement supérieur par pays — variable discriminante importante pour le ML.

### 5.1 Fetching + Parsing

In [ ]:
URL_TERTIARY = "https://en.wikipedia.org/wiki/List_of_countries_by_tertiary_education_attainment"

soup_tert = fetch(URL_TERTIARY)
tables_tert = soup_tert.find_all("table", class_="wikitable")
print(f"Tableaux trouvés : {len(tables_tert)}")

for i, tbl in enumerate(tables_tert[:3]):
    headers = [th.get_text(strip=True) for th in tbl.find_all("th")[:6]]
    print(f"  Tableau {i} : {headers}")

### 5.2 Extracting + Transforming

In [ ]:
df_tert_raw = extract_wikitable(tables_tert[0])
print("Colonnes brutes :", df_tert_raw.columns.tolist())
df_tert_raw.head()

In [ ]:
col_map_tert = {}
for col in df_tert_raw.columns:
    col_lower = str(col).lower()
    if "country" in col_lower or "nation" in col_lower:
        col_map_tert["country"] = col
    elif any(k in col_lower for k in ["total", "tertiary", "higher", "%", "percent", "attainment"]):
        if "tertiary_attainment_pct" not in col_map_tert:
            col_map_tert["tertiary_attainment_pct"] = col

print("Mapping colonnes détecté :", col_map_tert)

df_tert = pd.DataFrame()
df_tert["country"] = df_tert_raw[col_map_tert["country"]].apply(normalize_country)

if "tertiary_attainment_pct" in col_map_tert:
    df_tert["tertiary_attainment_pct"] = df_tert_raw[col_map_tert["tertiary_attainment_pct"]].apply(to_float)

df_tert = df_tert[df_tert["country"].str.strip() != ""].dropna()
df_tert = df_tert.reset_index(drop=True)

print(f"\nTertiary attainment : {len(df_tert)} pays")
df_tert.head(10)

---
## 6. Transforming — Fusion & construction du dataset final

On assemble tous les DataFrames extraits en un seul dataset propre,
avec le pays comme clé de jointure.

### 6.1 Aperçu des datasets collectés

In [ ]:
datasets = {
    "Education Index (Wikipedia)": df_edu,
    "Happiness Score (Wikipedia)": df_happy,
    "Literacy Rate (Wikipedia)": df_lit,
    "NationMaster Education": df_nm,
    "Tertiary Attainment (Wikipedia)": df_tert,
}

print(f"{'Dataset':<40} {'Pays':>6} {'Colonnes'}")
print("-" * 70)
for name, df in datasets.items():
    print(f"{name:<40} {len(df):>6}   {[c for c in df.columns if c != 'country']}")

### 6.2 Fusion progressive (outer join sur 'country')

In [ ]:
def make_join_key(name: str) -> str:
    """Clé de jointure robuste : lowercase, strip, suppression des accents courants
    et des caractères non-alpha pour absorber les variantes typographiques entre sources.
    
    Exemples : "Côte d'Ivoire" == "Cote d Ivoire", "Korea, South" == "Korea  South"
    """
    s = str(name).lower().strip()
    # Accents les plus fréquents dans les noms de pays
    replacements = {
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "à": "a", "â": "a", "ä": "a", "á": "a",
        "î": "i", "ï": "i", "í": "i",
        "ô": "o", "ö": "o", "ó": "o",
        "û": "u", "ü": "u", "ú": "u",
        "ç": "c", "ñ": "n", "ß": "ss",
        "'": " ", "'": " ", "-": " ", ",": " ", ".": "",
    }
    for char, repl in replacements.items():
        s = s.replace(char, repl)
    # Collapse des espaces multiples
    s = re.sub(r"\s+", " ", s).strip()
    return s


def add_join_key(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["_key"] = df["country"].apply(make_join_key)
    return df


# Déduplication préventive + ajout de la clé de jointure
dfs_to_merge = []
for name, df in datasets.items():
    if df.empty:
        print(f"  [SKIP] {name} — vide")
        continue
    df_clean = add_join_key(df.drop_duplicates(subset="country"))
    dfs_to_merge.append((name, df_clean))
    print(f"  [OK]   {name} — {len(df_clean)} pays uniques")

# Fusion séquentielle sur la clé normalisée
df_merged = dfs_to_merge[0][1]
for name, df_next in dfs_to_merge[1:]:
    before = len(df_merged)
    df_merged = pd.merge(df_merged, df_next, on="_key", how="outer", suffixes=("", f"_dup"))
    # Résolution des colonnes 'country' dupliquées : garder la valeur non-nulle
    dup_country = "country_dup"
    if dup_country in df_merged.columns:
        df_merged["country"] = df_merged["country"].fillna(df_merged[dup_country])
        df_merged = df_merged.drop(columns=[dup_country])
    print(f"  Après merge '{name}' : {before} → {len(df_merged)} lignes")

# Supprimer la clé de jointure technique
df_merged = df_merged.drop(columns=["_key"])

print(f"\nDataset fusionné : {df_merged.shape[0]} pays × {df_merged.shape[1]} colonnes")
print("Colonnes :", df_merged.columns.tolist())

### 6.3 Nettoyage final

In [ ]:
# Pays avec trop de valeurs manquantes (> 80% des features)
feature_cols = [c for c in df_merged.columns if c != "country"]
n_features = len(feature_cols)

df_merged["missing_rate"] = df_merged[feature_cols].isnull().sum(axis=1) / n_features

print(f"Distribution des taux de valeurs manquantes par pays :")
print(df_merged["missing_rate"].describe())

# Seuil : garder les pays avec au moins 2 features renseignées
threshold = 1 - (2 / max(n_features, 1))
df_filtered = df_merged[df_merged["missing_rate"] <= threshold].copy()
df_filtered = df_filtered.drop(columns=["missing_rate"])
df_filtered = df_filtered.sort_values("country").reset_index(drop=True)

print(f"\nAprès filtrage : {len(df_filtered)} pays")
print(f"Pays exclus    : {len(df_merged) - len(df_filtered)} (trop de NaN)")

In [ ]:
# Rapport de complétude par colonne
completeness = pd.DataFrame({
    "feature": feature_cols,
    "n_renseigné": [df_filtered[c].notna().sum() for c in feature_cols],
    "pct_renseigné": [round(df_filtered[c].notna().mean() * 100, 1) for c in feature_cols],
})
print(completeness.to_string(index=False))

### 6.4 Aperçu du dataset final

In [ ]:
print(f"Shape final : {df_filtered.shape}")
df_filtered.head(20)

In [ ]:
# Statistiques descriptives
df_filtered.describe()

---
## 7. Export CSV

In [ ]:
OUTPUT_PATH = "happiness_education_dataset.csv"

df_filtered.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"CSV exporté : {OUTPUT_PATH}")
print(f"  {len(df_filtered)} lignes (pays)")
print(f"  {len(df_filtered.columns)} colonnes : {df_filtered.columns.tolist()}")

# Vérification de l'import
df_check = pd.read_csv(OUTPUT_PATH)
print(f"\nVérification lecture : {df_check.shape} — OK")

---
## Annexe — Diagnostic & debug

Cellules utiles si des colonnes ne sont pas détectées automatiquement.

In [ ]:
# Inspecter toutes les colonnes d'un DataFrame brut
def inspect_df(df, name=""):
    print(f"=== {name} ===")
    print(f"Shape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print(df.head(3).to_string())
    print()

# Exemple d'utilisation :
# inspect_df(df_edu_raw, "Education Index brut")
# inspect_df(df_happy_raw, "Happiness brut")

In [ ]:
# Vérifier les pays présents dans une source mais absents d'une autre
def diff_countries(df_a, df_b, name_a="A", name_b="B"):
    set_a = set(df_a["country"].dropna())
    set_b = set(df_b["country"].dropna())
    only_a = sorted(set_a - set_b)
    only_b = sorted(set_b - set_a)
    print(f"Dans {name_a} mais pas {name_b} ({len(only_a)}) : {only_a[:20]}")
    print(f"Dans {name_b} mais pas {name_a} ({len(only_b)}) : {only_b[:20]}")

# Exemple :
# diff_countries(df_edu, df_happy, "Education", "Happiness")